# Chapter 11: Production Systems & MLOps for AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abda-dc/ai-engineering-zero-to-hero/blob/main/book/chapters/part4-agents-prod/ch11-production-systems-and-mlops.ipynb)

**Part:** Part IV: Autonomous Agents, MCP & Production MLOps  
**Chapter Identifier:** `ch11`  

## Core Concepts Covered in this Chapter:
- **System Design**
- **Microservices**
- **Redis Caching**
- **Celery/Queues**
- **Model Serving with vLLM/FastAPI**
- **Docker/Compose**
- **GitHub Actions CI/CD**
- **Prometheus/Grafana Monitoring**

---


## 1. Production AI System Architecture & Microservices

Production AI infrastructure decouples high-throughput API gateways from compute-heavy model serving engines (vLLM, Ollama, Triton) using asynchronous queues (Redis/Celery) and semantic caching layers.

```mermaid
flowchart TD
    Client[Web & Mobile Clients] --> Gateway[FastAPI API Gateway]
    Gateway --> Cache{Redis Semantic Cache}
    Cache -->|Cache Hit| ReturnFast[Return in 5ms]
    Cache -->|Cache Miss| Queue[Redis Task Queue / Celery]
    Queue --> Serving[vLLM / Ollama Cluster]
    Serving --> Vector[pgvector / Qdrant Storage]
    Serving --> Gateway
    Gateway --> Telemetry[Prometheus & Grafana]
```


In [ ]:
print("Production AI System: Decoupled API Gateway, Semantic Cache, Async Serving, and Telemetry.")

## 2. Exact & Semantic Caching with Redis

Exact caching stores responses keyed by prompt MD5 hash. Semantic caching embeds the incoming query and performs vector similarity against cached queries, returning the cached response if similarity exceeds threshold (e.g. 0.96), saving 90%+ latency and cost.


In [ ]:
class SemanticCacheSimulator:
    def __init__(self, threshold: float = 0.90):
        self.cache = []
        self.threshold = threshold
        self.hits = 0
        self.misses = 0

    def get(self, query_vec: list) -> str:
        for item in self.cache:
            sim = sum(q * c for q, c in zip(query_vec, item["vector"]))
            if sim >= self.threshold:
                self.hits += 1
                return f"[CACHE HIT: {sim:.2f}] {item['response']}"
        self.misses += 1
        return None

    def put(self, query_vec: list, response: str):
        self.cache.append({"vector": query_vec, "response": response})

cache = SemanticCacheSimulator(threshold=0.92)
cache.put([1.0, 0.0, 0.0], "LangGraph orchestrates cyclical agent state graphs.")

hit_result = cache.get([0.98, 0.1, 0.0])
miss_result = cache.get([0.1, 0.9, 0.0])

print("Query 1 Result:", hit_result)
print("Query 2 Result:", miss_result)
print(f"Stats: Hits={cache.hits}, Misses={cache.misses}")

## 3. High-Throughput Serving & Streaming with FastAPI

LLM generation is slow (seconds to minutes). Real-time UX demands Server-Sent Events (SSE) or WebSockets so users see tokens stream immediately as they are sampled.


In [ ]:
import json

def simulate_token_stream(prompt: str):
    tokens = ["Deep", " Learning", " powers", " modern", " intelligent", " agents."]
    for idx, tok in enumerate(tokens):
        sse_event = f"data: {json.dumps({'index': idx, 'token': tok})}\n\n"
        yield sse_event

print("Simulated SSE Stream:")
for chunk in simulate_token_stream("Hello"):
    print(chunk.strip())

---
## Hands-On Capstone Lab: Build a Resilient API Gateway with Rate Limiting & Prometheus Metrics

### Objective:
Build an API Gateway middleware class that enforces token bucket rate limiting and tracks Prometheus metrics (request counts, latency histograms).

### Instructions:
1. Implement metrics simulator.
2. Record request latency and status codes.
3. Verify telemetry summary metrics.


In [ ]:
class PrometheusMetricsSimulator:
    def __init__(self):
        self.requests_total = {}
        self.latencies = []

    def record_request(self, status: int, duration_ms: float):
        self.requests_total[status] = self.requests_total.get(status, 0) + 1
        self.latencies.append(duration_ms)

    def summary(self) -> dict:
        avg_lat = sum(self.latencies) / max(len(self.latencies), 1)
        return {
            "requests_by_status": self.requests_total,
            "total_requests": len(self.latencies),
            "avg_latency_ms": round(avg_lat, 2)
        }

metrics = PrometheusMetricsSimulator()
metrics.record_request(200, 14.2)
metrics.record_request(200, 18.5)
metrics.record_request(429, 1.1)

print("Prometheus Telemetry Summary:", metrics.summary())

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: What is the primary benefit of Semantic Caching over standard Exact-Match Caching for LLMs?
- **A)** It converts text into MP3 audio
- **B)** It recognizes paraphrased queries with identical semantic intent and serves cached responses without querying the LLM
- **C)** It eliminates the need for a web server
- **D)** It only works with SQL databases

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) It recognizes paraphrased queries with identical semantic intent and serves cached responses without querying the LLM

**Explanation:** Users ask identical questions with slight variations in wording. Semantic caching computes vector similarity against cached queries, serving answers instantly when intent matches.
</details>

### Question 2: Why is Server-Sent Events (SSE) streaming essential in generative AI web interfaces?
- **A)** It drastically reduces Time-To-First-Token (TTFT) perception by rendering tokens to the user as they are generated
- **B)** It prevents the user from closing the browser window
- **C)** It reduces model size by 4x
- **D)** It eliminates the need for backend servers

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) It drastically reduces Time-To-First-Token (TTFT) perception by rendering tokens to the user as they are generated

**Explanation:** Without streaming, users must wait for the entire multi-second response to complete. Streaming renders partial tokens immediately, maximizing user perceived responsiveness.
</details>

### Question 3: What is the role of Prometheus in an MLOps monitoring stack?
- **A)** To generate synthetic training data
- **B)** To scrape and store time-series metrics (request rates, latency histograms, GPU memory, error codes)
- **C)** To deploy Docker containers to Kubernetes
- **D)** To run unit tests automatically

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) To scrape and store time-series metrics (request rates, latency histograms, GPU memory, error codes)

**Explanation:** Prometheus is a time-series monitoring system that scrapes metrics endpoints at regular intervals and enables alerting based on SLO thresholds.
</details>

### Question 4: What serving optimization is used by engines like vLLM to dramatically increase throughput?
- **A)** PagedAttention: managing KV cache memory like OS virtual memory paging to eliminate fragmentation
- **B)** Deleting half of the model layers
- **C)** Running models without an operating system
- **D)** Translating all prompts into binary machine code

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) PagedAttention: managing KV cache memory like OS virtual memory paging to eliminate fragmentation

**Explanation:** PagedAttention treats the KV cache as non-contiguous memory pages (inspired by OS virtual memory), reducing memory waste to near zero and boosting batch throughput 2-4x.
</details>
